In [44]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

In [45]:
df = pd.read_csv(".\X_train\clinical_train.csv")
gene_df = pd.read_csv('.\X_train\molecular_train.csv')
target_df = pd.read_csv('target_train.csv')

In [46]:
df['ID'] = df['ID'].astype(str)
gene_df['ID'] = df['ID'].astype(str)


In [47]:
temp = df.merge(gene_df,how='left',on=['ID'])

In [48]:
temp = temp[['ID','EFFECT']]
wide = pd.crosstab(temp['ID'],temp['EFFECT']).reset_index()

cols = wide.columns.drop('ID')
wide[cols] = (wide[cols]>0).astype(int)

In [49]:
wide.head()

GENE,ID,ARID1A,ARID2,ASXL1,ASXL2,ATRX,BCOR,BCORL1,BRAF,BRCC3,...,TERT,TET2,TP53,U2AF1,U2AF2,WT1,ZBTB33,ZMYM3,ZNF318,ZRSR2
0,P100000,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,P100001,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,P100002,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,P100004,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,P100005,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [50]:
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
import numpy as np

In [51]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)


OS_STATUS    float64
OS_YEARS     float64
dtype: object


In [52]:
X = wide.loc[wide['ID'].isin(target_df['ID'])]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [54]:
y_train_struct = Surv.from_arrays(event=(y_train['OS_STATUS'] == 1),time=y_train['OS_YEARS'])
features = X_train.columns.drop('ID')

In [55]:
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01, max_iter=1000)
cox_lasso.fit(X_train[features], y_train_struct)


C:\Users\sheno\Downloads\soton_y3\challenge\VENV\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\sheno\Downloads\soton_y3\challenge\VENV\Lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The CoxnetSurvivalAnalysis or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
C:\Users\sheno\Downloads\soton_y3\challenge\VENV\Lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The

CoxnetSurvivalAnalysis(alpha_min_ratio=0.01, l1_ratio=1.0, max_iter=1000)

In [56]:
coefs = pd.Series(cox_lasso.coef_[:, 0], index=features)
selected_features = coefs[coefs != 0].index.tolist()
print("Selected mutations:", selected_features)



Selected mutations: []
